<a href="https://colab.research.google.com/github/Soumit2k05/MP-1-7th-Sem-Soumit-Subhasish-Roshan-Rajaram/blob/main/03_model_training_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install scikit-learn xgboost joblib

Step 1 — Load the feature-engineered dataset

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [3]:
df = pd.read_csv("PJME_hourly.csv", parse_dates=['Datetime'], index_col='Datetime')
df = df.sort_index().dropna()

df['Hour']    = df.index.hour
df['Day']     = df.index.day
df['Month']   = df.index.month
df['Weekday'] = df.index.weekday
df['Weekend'] = (df.index.dayofweek >= 5).astype(int)
df['Quarter'] = df.index.quarter
df['Year']    = df.index.year
df['Lag_1']   = df['PJME_MW'].shift(1)
df['Lag_24']  = df['PJME_MW'].shift(24)
df['Lag_168'] = df['PJME_MW'].shift(168)
df['Rolling_Mean_24']  = df['PJME_MW'].rolling(24).mean()
df['Rolling_Mean_168'] = df['PJME_MW'].rolling(168).mean()
df.dropna(inplace=True)

print(f"Shape: {df.shape}")
df.head()

Shape: (145198, 13)


,PJME_MW,Hour,Day,Month,Weekday,Weekend,Quarter,Year,Lag_1,Lag_24,Lag_168,Rolling_Mean_24,Rolling_Mean_168
Datetime,,,,,,,,,,,,,
2002-01-08 01:00:00,29445.0,1,8,1,1,0,1,2002,31187.0,26862.0,30393.0,33560.208333,32513.869048
2002-01-08 02:00:00,28670.0,2,8,1,1,0,1,2002,29445.0,25976.0,29265.0,33672.458333,32510.327381
2002-01-08 03:00:00,28375.0,3,8,1,1,0,1,2002,28670.0,25641.0,28357.0,33786.375000,32510.434524
2002-01-08 04:00:00,28542.0,4,8,1,1,0,1,2002,28375.0,25666.0,27899.0,33906.208333,32514.261905
2002-01-08 05:00:00,29261.0,5,8,1,1,0,1,2002,28542.0,26328.0,28057.0,34028.416667,32521.428571


In [4]:
# Resample from minute-level to hourly averages
# Brings 2 million rows down to ~35,000 — much faster to train on
df = df.resample('h').mean()
df = df.dropna() # Drop any NaNs introduced by resampling

# Engineer time features
df['Hour']       = df.index.hour
df['Day']        = df.index.day
df['Month']      = df.index.month
df['Weekday']    = df.index.dayofweek
df['Is_weekend'] = (df.index.dayofweek >= 5).astype(int)

In [6]:
TARGET = 'PJME_MW'

FEATURES = [
    'Hour', 'Day', 'Month', 'Weekday', 'Weekend',
    'Quarter', 'Year', 'Lag_1', 'Lag_24', 'Lag_168',
    'Rolling_Mean_24', 'Rolling_Mean_168'
]

X = df[FEATURES]
y = df[TARGET]

Step 2 — Define features (X) and target (y)

In [7]:
# Target
TARGET = 'Global_active_power'

In [8]:

#Features
FEATURES = ['Hour', 'Day', 'Month', 'Weekday', 'Weekend',
            'Quarter', 'Year', 'Lag_1', 'Lag_24', 'Lag_168',
            'Rolling_Mean_24', 'Rolling_Mean_168']
TARGET = 'PJME_MW'

X = df[FEATURES]
y = df[TARGET]

Step 3 — Chronological split (the critical step)


In [9]:
SPLIT_DATE = '2015-01-01'
train = df[df.index < SPLIT_DATE]
test  = df[df.index >= SPLIT_DATE]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

Train: 113,757 rows | Test: 31,437 rows


In [10]:
#drop missing value
train = X_train.copy()
train['target'] = y_train

train = train.dropna()

X_train = train.drop('target', axis=1)
y_train = train['target']

Step 4 — Train all three models

In [11]:
# Model 1: Linear Regression (baseline)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Model 3: XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, random_state=42,
                   early_stopping_rounds=20, eval_metric='rmse')

xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb.predict(X_test)


Step 5 — Compute all evaluation metrics

In [12]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name:25s}  MAE={mae:.2f} MW  |  RMSE={rmse:.2f} MW  |  R²={r2:.4f}')

evaluate('Linear Regression', y_test, lr_pred)
evaluate('Random Forest',     y_test, rf_pred)
evaluate('XGBoost',           y_test, xgb_pred)

Linear Regression          MAE=964.99 MW  |  RMSE=1232.73 MW  |  R²=0.9635
Random Forest              MAE=334.93 MW  |  RMSE=464.87 MW  |  R²=0.9948
XGBoost                    MAE=353.03 MW  |  RMSE=467.30 MW  |  R²=0.9947


Step 6 — Save the best model for the dashboard

In [13]:
import joblib
import os

# Create the 'dashboard' directory if it doesn't exist
os.makedirs('dashboard', exist_ok=True)

# Save Random Forest (or XGBoost — whichever scores best)
joblib.dump(rf, 'dashboard/rf_model.pkl')
joblib.dump(xgb, 'dashboard/xgb_model.pkl')
print('Models saved.')

Models saved.
